<a href="https://colab.research.google.com/github/tekpinar/deepflex/blob/main/flexibilitiesFromAlphaFold3Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---

# **Calculate Protein Flexibility from AlphaFold3 Models**

---
Attention: If you run some cells multiple times, we highly recommend you to clean all files and restart! You can go to the dropdown menu 'Disconnect and delete runtime' (under the Runtime menu) for a complete fresh restart.


# Install all required libraries

In [ ]:
!pip install py3dmol
!pip install biopython

# Upload your zip file that you downloaded from AlphaFold3 server

In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload your zip file:")
uploaded = files.upload()

# Get the first uploaded file name
if uploaded:
    fn = list(uploaded.keys())[0]
    print(f"File '{fn}' uploaded successfully.")
else:
    print("No file uploaded.")
    fn = None

# Unzip the data
Next, we will unzip the uploaded file. It will extract its contents to the current directory and then attempt to identify the name of the main extracted folder, storing it in the `extract_dir` variable. This variable can then be used in subsequent steps.

In [ ]:
# Unzip the file and determine the extracted directory name
if fn and fn.endswith('.zip'):
    try:
        # Get directories before unzipping to identify new ones created by extraction
        dirs_before = set([d for d in os.listdir('.') if os.path.isdir(d) and d not in ['sample_data', '.config']])

        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"Successfully unzipped '{fn}'.")

        # Get directories after unzipping
        dirs_after = set([d for d in os.listdir('.') if os.path.isdir(d) and d not in ['sample_data', '.config']])

        # Find new directories created during unzipping
        new_dirs = list(dirs_after - dirs_before)

        if new_dirs:
            extract_dir = None
            # Try to find a directory that closely matches the original zip file name
            base_name = os.path.splitext(os.path.basename(fn))[0]
            # Remove common suffixes like '(1)' if present due to re-upload (e.g., file (1).zip -> file)
            if base_name.endswith(' (1)'):
                base_name = base_name[:-4] # Remove ' (1)'

            for d in new_dirs:
                if d == base_name:
                    extract_dir = d
                    break

            if not extract_dir and len(new_dirs) == 1:
                extract_dir = new_dirs[0]
                print(f"Using the only new directory found: '{extract_dir}'")
            elif not extract_dir:
                # Fallback to the first new directory if no good match and multiple new dirs
                extract_dir = new_dirs[0]
                print(f"Warning: Multiple new directories found, and no exact match for '{base_name}'. Using first new directory: '{extract_dir}'")

            print(f"Extracted directory identified: '{extract_dir}'")
        else:
            # Fallback if no *new* directories are identified but the base name directory exists
            base_name = os.path.splitext(os.path.basename(fn))[0]
            if base_name.endswith(' (1)'):
                base_name = base_name[:-4] # Remove ' (1)'

            if os.path.isdir(base_name):
                extract_dir = base_name
                print(f"No new directory found, but a directory matching the zip file name '{extract_dir}' exists. Assuming this is the extracted directory.")
            else:
                extract_dir = None
                print("No new directory found after extraction, and no directory matching the zip file name was found.")

    except zipfile.BadZipFile:
        print(f"Error: '{fn}' is not a valid zip file.")
        extract_dir = None
    except Exception as e:
        print(f"An error occurred during unzipping: {e}")
        extract_dir = None
else:
    print("No valid zip file was uploaded for unzipping, or no file was uploaded.")
    extract_dir = None

# # The name of the extracted folder is now stored in 'extract_dir'
# if extract_dir:
#     print(f"The name of the extracted folder is stored in the variable `extract_dir`: '{extract_dir}'")

# Create some functions that will be used later

In [ ]:
from scipy.stats import rankdata,zscore

def rankSortData(dataArray):
    """
        This function ranksorts protein data and converts it
        to values between [0,1.0].

    Parameters:
    ----------
    dataArray: numpy array of arrays
               data read by numpy.loadtxt

    Returns:
    -------
    normalizedRankedDataArray: numpy array
    """
    normalizedRankedDataArray = rankdata(dataArray)/float(len(dataArray))

    return (normalizedRankedDataArray)

def get_bfactor_range_v2(pdb_file):
    """
    Extract B-factor values from PDB file and return their range

    Parameters:
    pdb_file (str): Path to the PDB file

    Returns:
    tuple: (minimum B-factor, maximum B-factor)
    """
    bfactors = []
    # print(pdb_file[0])
    allLines = pdb_file[0].split('\n')
    for line in allLines:
      # print(line)
      if line.startswith('ATOM') or line.startswith('HETATM'):
          try:
            # B-factor is typically in columns 61-66
            bfactor = float(line[60:66].strip())
            bfactors.append(bfactor)
          except (ValueError, IndexError):
            continue

    if not bfactors:
      return (0, 100)  # default range if no B-factors found

    return (min(bfactors), max(bfactors))

def visualize_protein_bfactor(pdb_file):
    """
    Visualize protein structure in cartoon representation colored by B-factor
    using automatically determined range and rainbow colors

    Parameters:
    pdb_file (str): Path to the PDB file
    """

    # Create a py3Dmol view instance
    view = py3Dmol.view()

    # Get B-factor range from the file
    bfactor_min, bfactor_max = get_bfactor_range_v2(pdb_file)

    # # Load the PDB file
    # with open(pdb_file, 'r') as f:
    #     pdb_data = f.read()

    # # Add the molecule to the viewer
    # view.addModel(pdb_data, "pdb")

    for pdb_string in pdb_file:
      view.addModel(pdb_string, 'pdb')

    # Set cartoon representation with rainbow coloring based on B-factor
    view.setStyle({'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'linear',  # Using rainbow color scheme
            'min': bfactor_min,
            'max': bfactor_max,
            'colors': ["blue", "white", "red"]
        }
    }})

    # Center and zoom the view
    view.zoomTo()

    # Add legend for B-factor coloring
    view.addPropertyLabels(
        prop='b',
        gradient='bwr',
        min=bfactor_min,
        max=bfactor_max,
        legend={'x': 0.85, 'y': 0.5}
    )

    # # Add text showing the B-factor range
    # view.addLabel(f"B-factor range: {bfactor_min:.2f} - {bfactor_max:.2f}",
    #              {'position': {'x': -20, 'y': -20, 'z': 0},
    #               'backgroundColor': 'white',
    #               'fontColor': 'black'})

    return view


# Calculate MSF from AlphaFold3 models

In [ ]:
from Bio.PDB import MMCIFParser, Selection, Superimposer
import numpy as np
import os
import pandas as pd
import glob

alphafold3_structure_dir = './'
print(f"AlphaFold3 structure directory: {alphafold3_structure_dir}")
alphafold3_msf_data = []
cif_parser = MMCIFParser(QUIET=True)

# Find and order cif files according to model number
cif_files = glob.glob('*.cif')
cif_files.sort(key=lambda x: int(os.path.basename(x).split('_model_')[1].split('.cif')[0]))
print(cif_files)

if len(cif_files) != 5:
  print(f"Warning: Found {len(cif_files)} CIF files expected to be 5. ")
  sys.exit(-1)

# Order cif files according to model number
cif_files.sort(key=lambda x: int(os.path.basename(x).split('_model_')[1].split('.cif')[0]))

# The original code had 'continue' here, which is a syntax error outside a loop.
# Since `len(cif_files)` is 5, this block would not execute anyway for this input.
# If it were intended to stop processing, it should be an early exit or raise.
if len(cif_files) != 5:
    print(f"Warning: Found {len(cif_files)} CIF files, expected 5. Cannot proceed with MSF calculation for this protein.")
    # Set alphafold3_msf_data to None or an empty list and exit if processing cannot continue
    alphafold3_msf_data = [] # Ensure it's empty if cannot proceed
    # An alternative would be to return from a function or raise an exception.
else:
    print(cif_files)
    protein_id_lower='test_protein'
    protein_id = 'test_protein'
    try:
        all_ca_atoms = []

        # Parse each CIF file and collect CA atoms
        for cif_file_path in cif_files:
            # Use a unique structure ID for each parsed file
            structure_id = os.path.splitext(os.path.basename(cif_file_path))[0]
            structure = cif_parser.get_structure(structure_id, cif_file_path)

            # Extract C-alpha atoms from the first (and usually only) model in a CIF file
            model = structure[0]
            ca_atoms = [atom for atom in Selection.unfold_entities(model, 'A') if atom.get_id() == 'CA']

            if not ca_atoms:
                print(f"Warning: No C-alpha atoms found in {cif_file_path}. Skipping this model.")
                # If a model has no CA atoms, it will cause issues later, so we should skip.
                # For consistency, it might be better to exit the whole calculation here.
                raise ValueError(f"No C-alpha atoms found in one of the models: {cif_file_path}")

            all_ca_atoms.append(ca_atoms)

        if not all_ca_atoms or not all_ca_atoms[0]:
            print(f"Warning: No C-alpha atoms found for protein {protein_id} in AlphaFold3 structures. Skipping MSF calculation.")
            # This will ensure alphafold3_msf_data remains empty if no CA atoms are found.
        else:
            # Ensure all models have the same number of C-alpha atoms for alignment
            num_ca_per_model = [len(ca_list) for ca_list in all_ca_atoms]
            if not all(n == num_ca_per_model[0] for n in num_ca_per_model):
                print(f"Warning: Inconsistent number of C-alpha atoms across AlphaFold3 models for {protein_id}. Skipping MSF calculation.")
            else:
                num_residues = num_ca_per_model[0]

                # Structural alignment to the first model
                superimposer = Superimposer()
                mobile_coords = []
                fixed_coords = np.array([atom.get_coord() for atom in all_ca_atoms[0]])

                for i, ca_atoms_model in enumerate(all_ca_atoms):
                    if i == 0:
                        mobile_coords.append(fixed_coords)
                        continue

                    mobile_atoms = ca_atoms_model
                    fixed_atoms = all_ca_atoms[0]

                    fixed_atoms_map = []
                    mobile_atoms_map = []
                    for fixed_atom, mobile_atom in zip(fixed_atoms, mobile_atoms):
                        if fixed_atom.get_parent().get_id() == mobile_atom.get_parent().get_id() and \
                            fixed_atom.get_parent().get_parent().get_id() == mobile_atom.get_parent().get_parent().get_id():
                            fixed_atoms_map.append(fixed_atom)
                            mobile_atoms_map.append(mobile_atom)

                    if len(fixed_atoms_map) != num_residues or len(mobile_atoms_map) != num_residues:
                        print(f"Warning: Mismatch in C-alpha atom count for alignment in AlphaFold3 {protein_id} model {i+1}. Using unaligned coordinates for this model.")
                        mobile_coords.append(np.array([atom.get_coord() for atom in ca_atoms_model]))
                    else:
                        superimposer.set_atoms(fixed_atoms_map, mobile_atoms_map)
                        superimposer.apply(mobile_atoms)
                        mobile_coords.append(np.array([atom.get_coord() for atom in mobile_atoms]))

                mobile_coords = np.array(mobile_coords)

                # Calculate MSF
                if len(cif_files) > 1:
                    sq_fluctuations = np.sum(np.square(mobile_coords - np.mean(mobile_coords, axis=0)), axis=(0, 2)) / (len(cif_files) - 1)
                else:
                    sq_fluctuations = np.zeros(num_residues)

                # Ranksort normalize MSF values
                if len(sq_fluctuations) > 1:
                    normalized_msf = rankSortData(sq_fluctuations)
                else:
                    normalized_msf = sq_fluctuations # If only one value, no normalization needed

                # Store results
                # Ensure residue_numbers are correctly extracted from the first model's CA atoms
                residue_numbers = [atom.get_parent().get_id()[1] for atom in all_ca_atoms[0]]
                for res_index_in_list, (res_num_pdb, msf_val) in enumerate(zip(residue_numbers, normalized_msf)):
                    alphafold3_msf_data.append({
                        'protein_id1': protein_id.split('_')[0],
                        'residue_number': res_num_pdb,
                        'residue_index': res_index_in_list + 1, # 1-based index
                        'normalized_msf': msf_val,
                        'source': 'AlphaFold3'
                    })

    except Exception as e:
        print(f"Error processing AlphaFold3 structures for {protein_id}: {e}")

alphafold3_msf_df = pd.DataFrame(alphafold3_msf_data)
#print(f"Processed MSF data for {len(alphafold3_msf_df['protein_id'].unique())} unique AlphaFold3 proteins.")
#print("AlphaFold3 MSF Data (first 5 rows):")
print(alphafold3_msf_df.head())
print(len(alphafold3_msf_df))

# The name of the extracted folder is now stored in 'extract_dir'
if extract_dir:
    print(f"The name of the extracted folder is stored in the variable `extract_dir`: '{extract_dir}'")
print(f"alphafold3_msf_data contains {len(alphafold3_msf_data)} entries after calculation.")


# Assign normalized MSF to Bfactors columns and write a pdb file for each model

In [ ]:
from Bio.PDB import PDBIO, MMCIFParser, Selection
import numpy as np
import os

cif_parser = MMCIFParser(QUIET=True)
protein_id_lower = 'test_protein' # Assuming this is available from previous context

# 1. Create a dictionary to map residue_index to normalized_msf values
msf_map = {}
if 'alphafold3_msf_data' in globals() and isinstance(alphafold3_msf_data, list):
    if not alphafold3_msf_data:
        print("Warning: `alphafold3_msf_data` is an empty list. All PDB B-factors will be set to 0.0.")
    else:
        for entry in alphafold3_msf_data:
            if 'residue_index' in entry and 'normalized_msf' in entry:
                msf_map[entry['residue_index']] = entry['normalized_msf']
            else:
                print(f"Warning: Entry in alphafold3_msf_data missing 'residue_index' or 'normalized_msf': {entry}")
else:
    print("Warning: `alphafold3_msf_data` is not defined or is not a list. All PDB B-factors will be set to 0.0.")


# 2. Loop through each cif_file in the cif_files list
for cif_file in cif_files:
    try:
        # 3. Parse the current cif_file
        structure_id = os.path.splitext(os.path.basename(cif_file))[0]
        structure = cif_parser.get_structure(structure_id, cif_file)

        # 4. Access the first model within the structure
        model = structure[0]

        # 5. For each residue in the model:
        for chain in model:
            for residue in chain:
                # Get the PDB residue number (e.g., from (' ', 1, ' ')) as the key
                # This matches the 'residue_index' if they are sequential and 1-based.
                res_num = residue.get_id()[1]

                # Look up the corresponding normalized_msf value from msf_map
                msf_value = msf_map.get(res_num, 0.0)
                if res_num not in msf_map:
                    # Only print warning if msf_map actually has data but this specific res_num is missing
                    if msf_map:
                        print(f"Info: MSF value for residue index {res_num} not found in map for {cif_file}. Setting B-factor to 0.0.")

                # Assign this normalized_msf value to the B-factor of *all* atoms in the residue
                # Biopython expects B-factors to be floats.
                for atom in residue.get_atoms():
                    atom.set_bfactor(float(msf_value))

        # 6. Construct the output PDB filename
        output_pdb_file = cif_file.replace('.cif', '.pdb')

        # 7. Create a PDBIO object
        io = PDBIO()

        # 8. Set the Structure object (which has been modified with B-factors) to the PDBIO object
        io.set_structure(structure)

        # 9. Write the modified structure to the output_pdb_file
        io.save(output_pdb_file)
        print(f"Successfully wrote modified PDB file: {output_pdb_file}")

    except Exception as e:
        print(f"Error processing {cif_file}: {e}")

print("Finished processing all CIF files and writing PDBs.")

# Verify creation of the pdb files

In [ ]:
import glob

# List all .pdb files to confirm their creation
pdb_files = glob.glob('*.pdb')
print("Newly created PDB files:")
for pdb_file in pdb_files:
    print(pdb_file)

if len(pdb_files) == 5:
    print(f"Confirmation: Successfully created {len(pdb_files)} PDB files.")
else:
    print(f"Warning: Expected 5 PDB files, but found {len(pdb_files)}.")

## Load PDB files and prepare for a structural alignment

Load each of the previously created PDB files into Biopython `Structure` objects. For each structure, extract its C-alpha atoms, which will be used for structural alignment. Store these structures for further processing.


In [ ]:
from Bio.PDB import PDBParser, Selection, Superimposer
import py3Dmol
import glob
import os
import io
parser = PDBParser(QUIET=True)
all_structures = []
all_ca_atoms_for_alignment = []

# Iterate through the pdb_files list
for pdb_file in pdb_files:
    try:
        # Parse the PDB file
        structure_id = os.path.splitext(os.path.basename(pdb_file))[0]
        structure = parser.get_structure(structure_id, pdb_file)

        # Append the parsed Structure object to all_structures
        all_structures.append(structure)

        # Extract C-alpha atoms from the first model
        model = structure[0]
        ca_atoms = [atom for atom in Selection.unfold_entities(model, 'A') if atom.get_id() == 'CA']

        # Append the list of C-alpha atoms for the current model
        all_ca_atoms_for_alignment.append(ca_atoms)

    except Exception as e:
        print(f"Error processing {pdb_file}: {e}")

print(f"Total structures loaded: {len(all_structures)}")
print(f"Total C-alpha atom lists loaded for alignment: {len(all_ca_atoms_for_alignment)}")

## Perform Structural Alignment
Perform a structural alignment of all loaded PDB models to the first model using Biopython's `Superimposer`. The `Superimposer` will modify the coordinates of the atoms in the mobile structures in place. After alignment, convert each aligned `Structure` object into a PDB formatted string.


In [ ]:
superimposer = Superimposer()
aligned_pdb_strings = []
io_object = PDBIO()

# Get the C-alpha atoms for the first structure (index 0) from all_ca_atoms_for_alignment
# These will serve as the fixed atoms for the alignment.
fixed_ca_atoms = all_ca_atoms_for_alignment[0]

# Convert the first (reference) structure to a PDB formatted string and add it to the list
# This ensures the reference structure is also included.
buffer = io.StringIO()
io_object.set_structure(all_structures[0])
io_object.save(buffer)
aligned_pdb_strings.append(buffer.getvalue())

# Iterate through the all_structures list along with their corresponding C-alpha atoms
# starting from the second structure (index 1) for alignment.
for i in range(1, len(all_structures)):
    mobile_structure = all_structures[i]
    mobile_ca_atoms = all_ca_atoms_for_alignment[i]

    # Set the fixed atoms and the mobile atoms to the Superimposer object
    superimposer.set_atoms(fixed_ca_atoms, mobile_ca_atoms)

    # Apply the calculated rotation and translation to the current mobile structure's atoms
    # The apply method modifies the coordinates of the atoms in the structure in place.
    superimposer.apply(mobile_structure.get_atoms())

    # Create an in-memory text buffer
    buffer = io.StringIO()

    # Set the modified structure to the PDBIO object
    io_object.set_structure(mobile_structure)

    # Save the structure to the in-memory buffer
    io_object.save(buffer)

    # Get the PDB formatted string from the buffer and append it to the aligned_pdb_strings list
    aligned_pdb_strings.append(buffer.getvalue())

print(f"Number of aligned PDB strings created: {len(aligned_pdb_strings)}")

## Visualize the aligned models with py3Dmol







In [ ]:
import py3Dmol
import pandas as pd
import re
from IPython.display import display

# Replace with your PDB file path
view = visualize_protein_bfactor(aligned_pdb_strings)
view.show()

# Plot 'Residue Numbers vs Normalized MSF' as an interactive 2D plot.

In [ ]:
import plotly.express as px

if 'alphafold3_msf_df' in globals() and not alphafold3_msf_df.empty:
    fig = px.line(
        alphafold3_msf_df,
        x='residue_number',
        y='normalized_msf',
        title='Normalized Mean Squared Fluctuation (MSF) vs. Residue Number',
        labels={'residue_number': 'Residue Number', 'normalized_msf': 'Normalized MSF'},
        hover_data={'residue_number': True, 'normalized_msf': ':.2f'}
    )
    fig.update_traces(mode='lines+markers', marker=dict(size=5))
    fig.update_layout(xaxis_title='Residue Number', yaxis_title='Normalized MSF')
    fig.show()
else:
    print("Error: alphafold3_msf_df is not available or is empty. Cannot generate plot.")

# Download the models with flexibilities at Bfactors as a zip file

In [ ]:
import zipfile
from google.colab import files
import glob
import os

# Identify all newly created PDB files
pdb_files = glob.glob('*.pdb')

# Define the name for the output zip file
zip_filename = 'alphafold3_flexibility_models.zip'

# Create a zip file and add all PDB files to it
with zipfile.ZipFile(zip_filename, 'w') as zf:
    for pdb_file in pdb_files:
        zf.write(pdb_file, os.path.basename(pdb_file))

print(f"Successfully created '{zip_filename}' containing {len(pdb_files)} PDB models.")

# Provide the download link to the user
files.download(zip_filename)